In [11]:
# splitting the dataset to : training , dev/validation , test
import torch 
import torch.nn.functional as F 
import matplotlib.pyplot as plt

words = open('names.txt','r').read().splitlines()
words[:8]


chars = sorted(list(set(''.join(words))))
stoi = {s : i+1 for i , s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}


In [12]:

def build_dataset(words):  
    block_size = 3 # context length: how many characters do we take to predict the next one?
    X, Y = [], []
    for w in words:

        #print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(itos[i] for i in context), '--->', itos[ix])
            context = context[1:] + [ix] # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [13]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 10), generator= g , requires_grad  = True)
W1 = torch.randn((30, 200), generator= g , requires_grad  = True)
b1 = torch.randn(200, generator= g , requires_grad  = True)
W2 = torch.randn((200, 27), generator= g , requires_grad  = True)
b2 = torch.randn(27, generator= g , requires_grad  = True)
parameters = [C, W1, b1, W2, b2]


sum(p.nelement() for p in parameters) # number of parameters in total

11897

In [14]:
for i in range(20000):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (32,))
  
  # forward pass
  emb = C[Xtr[ix]] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 200)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytr[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  #lr = lrs[i]
  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  #lri.append(lre[i])
 

#print(loss.item())

In [15]:
print(loss.item())

2.5709710121154785


In [16]:
emb = C[Xtr] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ytr)
loss

tensor(2.4322, grad_fn=<NllLossBackward0>)

In [17]:
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.4541, grad_fn=<NllLossBackward0>)